# Task B Run 31 — Gemma class-weight ablation

This is an evaluation-only experiment for the main remaining Task B weakness: the label distribution is highly uneven. The fixed classifier-training split contains 43.1% Gender, but only 5.9% Geo-political and 7.0% Violence. The current Gemma recipe uses aggressive inverse-frequency (balanced) weights.

This notebook compares:

| arm | loss weighting | seeds |
|---|---|---|
| balanced | current recipe: inverse-frequency class weights | 42, 43 |
| sqrt | square-root inverse-frequency weights | 42, 43 |

The sqrt setting is a gentler correction. It may improve macro-F1 by helping Geo-political and Violence without over-correcting the model toward rare classes. Focal loss is not used because it hurt Task B in the earlier MuRIL experiments.

The notebook averages the two seed probability matrices for each arm and uses a paired bootstrap on the same 530-row holdout. It creates no submission ZIP. If balanced predictions are attached from a previous Gemma run, they are reused and only the two sqrt models are trained.

Kaggle settings: Internet on, GPU T4 x2, and a Kaggle secret named HF_TOKEN if Hugging Face asks for authentication. Save Version → Save & Run All. Runtime is about 2–3 hours with an attached balanced baseline, or 3–4 hours standalone.


In [ ]:
import fnmatch, json, os, pathlib, re, subprocess, sys, time

T_START = time.time()
HARD_STOP = T_START + 11 * 3600
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

try:
    remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                            check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit('Cannot reach GitHub. Turn on Settings -> Internet, then restart.') from e

if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
print('HF_TOKEN available:', bool(os.environ.get('HF_TOKEN')))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('repo:', git('log', '-1', '--oneline'))
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path('/kaggle/working/b31_class_weight_outputs')
RUNS, LOGS = OUT / 'runs', OUT / 'logs'
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

MODEL = 'google/gemma-4-12B'
SPLIT = pathlib.Path(WORK) / 'data/derived/task_b_combined_holdout'
LABELS = ['Gender', 'Geo-political', 'Others', 'Political', 'Religion', 'Violence']
LABEL_COL = 'Hate Category'
train = pd.read_csv(SPLIT / 'train.csv')
holdout = pd.read_csv(SPLIT / 'holdout.csv')
assert (len(train), len(holdout)) == (3002, 530), (len(train), len(holdout))
HOLDOUT_FP = fingerprint(holdout['id'])
assert HOLDOUT_FP == 'f85f4f049b', HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(LABELS.index).to_numpy()
print(f'train {len(train)}, holdout {len(holdout)}, fingerprint {HOLDOUT_FP}')

def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(LABELS)), zero_division=0)
    macro = float(f1_score(y, pred, average='macro', zero_division=0))
    acc = float(accuracy_score(y, pred))
    print(f'{name:32s} macro-F1 {macro:.4f}  acc {acc:.4f}  | ' +
          '  '.join(f'{c[:5]} {f:.3f}' for c, f in zip(LABELS, per)))
    return {'macro_f1': macro, 'accuracy': acc, 'per_class_f1': per.tolist()}

def compare(name, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f'{name:32s} diff {r["diff"]:+.4f}  95% CI '
          f'[{r["ci95"][0]:+.4f}, {r["ci95"][1]:+.4f}]  P(better) {r["p_better"]:.2f}')
    return r

def load_probs(run_dir):
    run_dir = pathlib.Path(run_dir)
    p = run_dir / 'holdout_probs.npy'
    ids = run_dir / 'holdout_ids.csv'
    if not p.exists() or not ids.exists():
        return None
    got_ids = pd.read_csv(ids)['id'].tolist()
    if got_ids != holdout['id'].tolist():
        return None
    probs = np.load(p)
    if probs.shape != (len(holdout), len(LABELS)):
        return None
    return probs

def attached_dirs(names):
    found = {}
    if not os.path.isdir('/kaggle/input'):
        return found
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        if 'holdout_probs.npy' not in files:
            continue
        if pathlib.Path(root).name in names:
            found[pathlib.Path(root).name] = pathlib.Path(root)
    return found


In [ ]:
# Keep the Gemma stack separate from Kaggle's system transformers/peft versions.
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip',
                    '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, '-c',
                "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"],
               check=True)

def download_model():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt',
                                      'tokenizer*', '*.jinja'])

def llm_job(weight, seed):
    name = f'{weight}_s{seed}'
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task b --model {MODEL} '
           f'--prompt short --class-weight {weight} --train {SPLIT / "train.csv"} '
           f'--eval {SPLIT / "holdout.csv"} --predict {SPLIT / "holdout.csv"} '
           f'--out {RUNS / name} --seed {seed} --epochs 3 --lr 1e-4 --r 16 '
           f'--bs 8 --grad-accum 2 --eval-bs 16 --max-len 320 --cache {HF_CACHE}')
    return {'name': name, 'cmd': cmd, 'log': str(LOGS / f'{name}.log')}

def run_jobs(jobs):
    registry = {j['name']: j for j in jobs}
    retried = set()

    def on_done(name, code):
        job = registry[name]
        if not code and name not in retried:
            return []
        log_path = pathlib.Path(job['log'])
        log = log_path.read_text(errors='replace') if log_path.exists() else ''
        extra = ''
        if code == 4 and name not in retried:
            m = re.search(r'--seed (\d+)', job['cmd'])
            if m:
                extra = f' --seed {int(m.group(1)) + 1000}'
                print(f'{name}: collapse detected; retrying with alternate seed')
        elif 'OutOfMemoryError' in log and name not in retried:
            extra = ' --bs 4 --grad-accum 4 --eval-bs 8'
            print(f'{name}: OOM detected; retrying with smaller micro-batches')
        if extra:
            retried.add(name)
            retry = dict(job)
            retry['cmd'] = job['cmd'] + extra
            registry[name] = retry
            return [retry]
        return []
    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=on_done, stop_at=HARD_STOP)

download_model()


In [ ]:
# Reuse a compatible balanced Gemma run if it was attached; otherwise train the
# balanced control. The sqrt arm is always trained here.
prior_names = {
    'short_s42', 'short_s43', 'base_s42', 'base_s43',
    'gemma-4-12b_ho_s42', 'gemma-4-12b_ho_s43',
}
dirs = attached_dirs(prior_names)
balanced = {}
for seed in (42, 43):
    candidates = [name for name in (
        f'short_s{seed}', f'base_s{seed}', f'gemma-4-12b_ho_s{seed}'
    ) if name in dirs]
    for name in candidates:
        p = load_probs(dirs[name])
        if p is not None:
            balanced[seed] = p
            break
print('reused balanced seeds:', sorted(balanced))

jobs = [llm_job('sqrt', seed) for seed in (42, 43)]
jobs += [llm_job('balanced', seed) for seed in (42, 43) if seed not in balanced]
print('jobs:', [j['name'] for j in jobs])
codes = run_jobs(jobs)
print('exit codes:', codes)
print(f'elapsed {(time.time() - T_START) / 3600:.1f} h')


In [ ]:
def local_arm(weight):
    out = {}
    for seed in (42, 43):
        p = load_probs(RUNS / f'{weight}_s{seed}')
        if p is not None:
            out[seed] = p
    return out

sqrt = local_arm('sqrt')
reused_balanced_seeds = sorted(balanced)
balanced.update(local_arm('balanced'))
if set(balanced) != {42, 43}:
    raise RuntimeError(f'balanced arm incomplete: {sorted(balanced)}; inspect {LOGS}')
if set(sqrt) != {42, 43}:
    raise RuntimeError(f'sqrt arm incomplete: {sorted(sqrt)}; inspect {LOGS}')

for weight, arm in [('balanced', balanced), ('sqrt', sqrt)]:
    for seed, probs in sorted(arm.items()):
        report(y_ho, probs, f'{weight} seed {seed}')

balanced_mean = np.mean([balanced[42], balanced[43]], axis=0)
sqrt_mean = np.mean([sqrt[42], sqrt[43]], axis=0)
print()
scores = {
    'balanced_2seed_mean': report(y_ho, balanced_mean, 'balanced, 2 seeds'),
    'sqrt_2seed_mean': report(y_ho, sqrt_mean, 'sqrt, 2 seeds'),
}
print()
bootstrap = {'sqrt_minus_balanced': compare('sqrt mean - balanced mean', sqrt_mean, balanced_mean)}
use_sqrt = bootstrap['sqrt_minus_balanced']['p_better'] >= 0.7
print(f'Use sqrt for the final recipe: {use_sqrt}')

np.save(OUT / 'balanced_mean_holdout_probs.npy', balanced_mean)
np.save(OUT / 'sqrt_mean_holdout_probs.npy', sqrt_mean)
result = {
    'experiment': 'task_b_run31_gemma_class_weight_holdout',
    'commit': head,
    'holdout_rows': len(holdout),
    'holdout_fingerprint': HOLDOUT_FP,
    'reused_balanced_seeds': reused_balanced_seeds,
    'scores': scores,
    'bootstrap': bootstrap,
    'use_sqrt_for_final': use_sqrt,
    'exit_codes': codes,
    'elapsed_hours': (time.time() - T_START) / 3600,
}
json.dump(result, open(OUT / 'result.json', 'w'), indent=2)
print('Saved probability matrices and result.json under', OUT)
print('No submission ZIP is created.')
